In [1]:
import h5py
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
import requests

In [11]:
response = requests.get("https://goat.genomehubs.org/search?query=assembly_span%20AND%20tax_rank%28phylum%29&result=taxon&taxonomy=ncbi")
response.status_code
response.links



{}

In [139]:
with h5py.File("Data/cox1_perAA.h5", 'r') as h5_file:
    ids = list(h5_file.keys())
    embs = [np.array(h5_file[protein_id]) for protein_id in ids]


In [2]:
def pad_embedding(embs, max_size = 256):
    output = []
    for e in embs:
        e_len = e.shape[0]
        if e_len > max_size:
            raise ValueError('Embedding size larger than the padding size')
        pad_sz = max_size - e_len
        output.append(F.pad(torch.Tensor(e), (0, 0, 0, pad_sz)))
    return torch.stack(output)





In [169]:
embs = pad_embedding(embs)
dataset = TensorDataset(embs)

train_size = int(0.75 * len(dataset))  # 75% for training
val_size = len(dataset) - train_size  # 25% for validation

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)


In [154]:
def conv1d_block(in_channels, out_channels, kernel_size, padding, bn, act):
    # Define the convolutional layer
    layers = [nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)]

    # Add batch normalization if 'bn' is True
    if bn:
        layers.append(nn.BatchNorm1d(out_channels))

    # Add the activation function
    if act == 'relu':
        layers.append(nn.ReLU(inplace=True))
    elif act == 'lrelu':
        layers.append(nn.LeakyReLU(0.1, inplace=True))

    # Return the layers as a sequential model
    return nn.Sequential(*layers)

class DownsamplingResidualBlock(nn.Module):
    def __init__(self, num_filters, use_bn=True):
        super(DownsamplingResidualBlock, self).__init__()

        # First convolutional block with 1x1 kernel
        self.conv1 = conv1d_block(num_filters, num_filters//4, 1, padding=0,
                                  bn=use_bn, act='relu')

        self.proj_conv = conv1d_block(num_filters, num_filters//2, 1, padding=0,
                                  bn=use_bn, act='relu')
        #simple max pooling using a 3x3 window
        # Second convolutional block with 3x3 kernel
        self.conv2 = conv1d_block(num_filters//4, num_filters//4, 3, padding=1,
                                  bn=use_bn, act='relu')

        # Third convolutional block with 1x1 kernel and no activation function
        self.conv3 = conv1d_block(num_filters//4, num_filters//2, 1, padding=0,
                                  bn=use_bn, act=None)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # Pass the input through the convolutional blocks
        x_out = self.conv1(x)
        x_out = self.conv2(x_out)
        x_out = self.conv3(x_out)
        x_p = self.proj_conv(x)


        # Add the input to the output of the convolutional blocks
        return self.relu(x_p + x_out)


class Downsampler(nn.Module):
    def __init__(self, num_filters):
        super(Downsampler, self).__init__()

        layers = []
        while(num_filters >= 4):
            layers.append(DownsamplingResidualBlock(num_filters))
            num_filters = num_filters//2

        self.layers = nn.Sequential(*layers)
        self.final_layer = conv1d_block(2, 1, 1, padding=0,
                                  bn=True, act='relu')

    def forward(self, x):
        x = self.layers(x)
        return self.final_layer(x)


class simpleDownsampler(nn.Module):
    def __init__(self, num_filters, use_bn=True):
        super(simpleDownsampler, self).__init__()

        # First convolutional block with 1x1 kernel
        self.conv1 = conv1d_block(num_filters, 1, 1, padding=0,
                                  bn=use_bn, act='relu')

        self.proj_conv = conv1d_block(num_filters, num_filters, 1, padding=0,
                                  bn=use_bn, act='relu')
        #simple max pooling using a 3x3 window
        # Second convolutional block with 3x3 kernel
        self.conv2 = conv1d_block(1, 1, 3, padding=1,
                                  bn=use_bn, act='relu')

        # Third convolutional block with 1x1 kernel and no activation function
        self.conv3 = conv1d_block(1, num_filters, 1, padding=0,
                                  bn=use_bn, act='relu')

    def forward(self, x):
        
        x_out = self.conv1(x)
        x_out = self.conv2(x_out)
        x_out = self.conv3(x_out)
        x_p = self.proj_conv(x)
        return self.relu(x_p + x_out)

In [155]:
class UpsamplingResidualBlock(nn.Module):
    def __init__(self, num_filters, use_bn=True):
        super(UpsamplingResidualBlock, self).__init__()

        # First convolutional block with 1x1 kernel
        self.conv1 = conv1d_block(num_filters, num_filters*2, 1, padding=0,
                                  bn=use_bn, act=None)

        self.proj_conv = conv1d_block(num_filters, num_filters*2, 1, padding=0,
                                  bn=use_bn, act='relu')

        # Second convolutional block with 3x3 kernel
        self.conv2 = conv1d_block(num_filters*2, num_filters*2, 3, padding=1,
                                  bn=use_bn, act='relu')

        # Third convolutional block with 1x1 kernel and no activation function

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # Pass the input through the convolutional blocks
        x_out = self.conv1(x)
        x_out = self.conv2(x_out)
        x_p = self.proj_conv(x)


        # Add the input to the output of the convolutional blocks
        return self.relu(x_p + x_out)

class Upsampler(nn.Module):
    def __init__(self, num_filters, max_filters):
        super(Upsampler, self).__init__()

        layers = []
        while(num_filters < max_filters):
            layers.append(UpsamplingResidualBlock(num_filters))
            num_filters = num_filters*2

        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)

In [159]:
class BottleneckNet(nn.Module):
    def __init__(self, upsampler, downsampler):
        super(BottleneckNet, self).__init__()
        self.upsampler = upsampler
        self.downsampler = downsampler
    def forward(self, x):
        global_rep = self.downsampler(x)
        local_rep = self.upsampler(global_rep)
        return local_rep

In [175]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

down_m = Downsampler(256)
up_m = Upsampler(1, 256)
model = BottleneckNet(up_m, down_m)
model.to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
num_epochs = 100

for epoch in range(num_epochs):
    train_losses = 0.0
    val_losses = 0.0
    for batch in train_loader:
        inputs = batch[0].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, inputs)
        train_losses += loss.item()
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        for batch in val_loader:
            inputs = batch[0].to(device)
            outputs = model(inputs)
            val_loss = criterion(outputs, inputs)
            val_losses += val_loss.item()s

    print(f'Epoch [{epoch+1}/{num_epochs}], Training Loss: {train_losses/len(train_loader):.4f}, Validation Loss: {val_losses/len(val_loader):.4f}')



Epoch [1/100], Training Loss: 0.6319, Validation Loss: 0.3436
Epoch [2/100], Training Loss: 0.2327, Validation Loss: 0.1937
Epoch [3/100], Training Loss: 0.1454, Validation Loss: 0.1456
Epoch [4/100], Training Loss: 0.1121, Validation Loss: 0.1276
Epoch [5/100], Training Loss: 0.1029, Validation Loss: 0.1140
Epoch [6/100], Training Loss: 0.0969, Validation Loss: 0.1040
Epoch [7/100], Training Loss: 0.0835, Validation Loss: 0.0929
Epoch [8/100], Training Loss: 0.0790, Validation Loss: 0.0868
Epoch [9/100], Training Loss: 0.0742, Validation Loss: 0.0782
Epoch [10/100], Training Loss: 0.0680, Validation Loss: 0.0717
Epoch [11/100], Training Loss: 0.0627, Validation Loss: 0.0697
Epoch [12/100], Training Loss: 0.0598, Validation Loss: 0.0665
Epoch [13/100], Training Loss: 0.0586, Validation Loss: 0.0637
Epoch [14/100], Training Loss: 0.0585, Validation Loss: 0.0636
Epoch [15/100], Training Loss: 0.0580, Validation Loss: 0.0627
Epoch [16/100], Training Loss: 0.0577, Validation Loss: 0.0638
E

In [184]:
trained_downsampler = model.downsampler
trained_upsampler = model.upsampler
trained_upsampler.eval()
model.eval()
trained_downsampler.eval()
for batch in val_loader:
    input = batch[0].to(device)
    output = model(input)
    print(output[0], input[0])
    break

tensor([[0.1265, 0.0258, 0.0077,  ..., 0.0079, 0.0056, 0.0936],
        [0.1876, 0.0109, 0.0075,  ..., 0.0038, 0.0112, 0.0751],
        [0.0311, 0.0103, 0.0100,  ..., 0.0058, 0.0017, 0.0862],
        ...,
        [0.0072, 0.0027, 0.0025,  ..., 0.0003, 0.0188, 0.0267],
        [0.0403, 0.0045, 0.0029,  ..., 0.0000, 0.0053, 0.0193],
        [0.0120, 0.0188, 0.0035,  ..., 0.0003, 0.0094, 0.0313]],
       device='cuda:0', grad_fn=<SelectBackward0>) tensor([[ 0.1125, -0.2072, -0.1588,  ...,  0.0269,  0.1533,  0.1765],
        [-0.2913, -0.2025, -0.2026,  ..., -0.0642, -0.0584,  0.1776],
        [ 0.0652, -0.1138,  0.0483,  ..., -0.2315,  0.2140, -0.0743],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
       device='cuda:0')
